# Employee Compensation & Workforce Analytics
### Business Case Scenarios — Gold Layer Analysis

**Purpose:** This notebook builds a collection of analytics-ready gold tables that answer real-world HR and finance questions across school districts. Each scenario is self-contained — it reads from the **silver/gold** medallion layers, applies business logic, and persists the output as a Delta table in the gold zone.

**Data Sources:**
- `gold_employee_summary` — master employee record with derived fields (tenure, seniority, compensation totals)
- `gold_district_compensation` — district-level aggregated pay metrics
- `gold_certification_overview` — teacher certification lifecycle data
- `fact_pay_and_benefits` / `fact_pab_lineitem` / `dim_pab_item` — granular pay-and-benefits line items (silver layer)

**Scenarios Covered:**
| # | Scenario | Key Metric |
|---|----------|------------|
| 1 | Top 10 Highest Compensated per District | Total Compensation Rank |
| 2 | Promotion Pipeline Analysis | Eligible Count by Seniority |
| 3 | Admin vs Teacher Pay Equity | Pay Gap % |
| 4 | Overtime Cost Hotspots | OT % of Regular Pay |
| 5 | Certification Gap (Compliance Risk) | Days Since Expiry |
| 6 | Underpaid Senior Employees | Comp Gap vs District Median |
| 7 | Degree ROI Analysis | Premium vs Bachelor Baseline |
| 8 | Quarterly Pay Breakdown | Amount by Pay Type & Quarter |
| 9 | Benefits Cost Optimization | Benefits % of Total Comp |
| 10 | Retirement Risk & Workforce Planning | Headcount at Risk by Band |
| 11 | Teacher Certification Breadth | Cert Count & Compensation Correlation |

---
_Last refreshed: dynamically via `current_timestamp()` on each run._

In [0]:
# ── PySpark imports ──
# We pull in the standard toolkit: column functions, window specs, and type hints.
# These are used throughout the notebook for transformations, rankings, and schema control.

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, BooleanType, TimestampType

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# All data lives in Azure Data Lake under the "employee" container.
# Bronze = raw ingestion, Silver = cleansed/conformed, Gold = analytics-ready.

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema references ──
# Each layer has its own schema inside the employeedatacatalog catalog.

bronze_sch = "bronze_employee"
silver_sch = "silver_employee"
gold_sch = "gold_employee"
employee_db = "employeedatacatalog"

In [0]:
# ── Load the core tables we'll reference across all 11 scenarios ──
# Gold tables carry pre-computed fields like years_of_service, seniority_level,
# total_compensation, and promotion_eligible — so downstream logic stays clean.

gold_emp      = spark.table(f"{employee_db}.{gold_sch}.gold_employee_summary")       # master employee record
gold_district = spark.table(f"{employee_db}.{gold_sch}.gold_district_compensation")   # district-level comp summary
gold_cert     = spark.table(f"{employee_db}.{gold_sch}.gold_certification_overview")   # teacher cert lifecycle

# Silver-layer fact/dim tables for granular pay-line-item analysis (Scenario 8)
fact_pab      = spark.table(f"{employee_db}.{silver_sch}.fact_pay_and_benefits")
fact_lineitem = spark.table(f"{employee_db}.{silver_sch}.fact_pab_lineitem")
dim_pab_item  = spark.table(f"{employee_db}.{silver_sch}.dim_pab_item")

# Quick sanity check — row counts confirm data is loaded
print(f"gold_employee_summary:  {gold_emp.count():,} rows")
print(f"gold_certification:     {gold_cert.count():,} rows")

   
---
## Scenario 1 — Top 10 Highest Compensated Employees per District
**Business Question:** Who are the top earners in each school district, and what roles do they hold?

**Why it matters:** School boards and taxpayers often want transparency into how compensation dollars are distributed. Identifying the highest-paid employees per district helps surface potential outliers and ensures pay structures align with role responsibilities.

**Approach:**
- Rank employees within each district using a window function on `total_compensation`.
- Tag each employee as *Admin*, *Teacher*, or *Other* based on boolean flags.
- Persist the top 10 per district as a gold Delta table.

**Output table:** `gold_employee.top10_compensated_per_district`

In [0]:
# ============================================================
# SCENARIO 1: Top 10 Highest Compensated Employees per District
# ============================================================
# We use a window function to rank employees within each district
# by total_compensation (descending), then keep only the top 10.

# Step 1 — Define the ranking window: partition by district, order by pay
window_district = Window.partitionBy("DISTRICT_NAME").orderBy(col("total_compensation").desc())

# Step 2 — Rank, filter, and tag employee roles
df_top10_per_district = gold_emp \
    .filter(col("total_compensation").isNotNull()) \
    .withColumn("rank", rank().over(window_district)) \
    .filter(col("rank") <= 10) \
    .withColumn("role",                                    # Simplify admin/teacher flags into a readable label
        when(col("IS_ADMIN") == True, lit("Admin"))
        .when(col("IS_TEACHER") == True, lit("Teacher"))
        .otherwise(lit("Other"))) \
    .select(
        "DISTRICT_NAME", "rank", "EMP_ID", "EMPLOYEE_NAME", "role",
        "HIGHEST_EARNED_DEGREE", "years_of_service", "total_experience",
        "REG_PAY", "OVERTIME_PAY", "OTHER_PAY", "TOTAL_BENEFITS", "total_compensation"
    ) \
    .withColumn("_gold_ingested_at", current_timestamp())   # Audit column: when was this row written?

# Step 3 — Persist to gold Delta table
df_top10_per_district.write.format("delta").mode("overwrite").save(f"{gold_path}/top10_compensated_per_district")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.top10_compensated_per_district
    USING DELTA
    LOCATION '{gold_path}/top10_compensated_per_district'
""")

# Step 4 — Preview results ordered by district and rank
display(df_top10_per_district.orderBy("DISTRICT_NAME", "rank"))

   
---
## Scenario 2 — Promotion Pipeline Analysis
**Business Question:** How many employees are eligible for promotion in each district and seniority band? What does the pipeline look like?

**Why it matters:** A healthy promotion pipeline prevents bottlenecks and ensures institutional knowledge is passed on. If most eligible employees are clustered in one seniority tier, HR can proactively plan development programs.

**Approach:**
- Filter to employees where `promotion_eligible = True` (derived in the gold summary).
- Group by district and seniority level, then compute headcounts, avg tenure, avg compensation, and degree diversity.
- Order seniority bands logically from Entry → Senior.

**Output table:** `gold_employee.promotion_pipeline`

In [0]:
# ============================================================
# SCENARIO 2: Promotion Pipeline Analysis
# ============================================================
# Filters to promotion-eligible employees and groups them by
# district + seniority band to show the shape of the pipeline.

# Step 1 — Filter to eligible employees and aggregate
df_promotion_pipeline = gold_emp \
    .filter(col("promotion_eligible") == True) \
    .groupBy("DISTRICT_NAME", "seniority_level") \
    .agg(
        count("EMP_ID").alias("eligible_count"),                          # How many are eligible?
        round(avg("years_of_service"), 1).alias("avg_years_of_service"),   # Avg tenure of eligible pool
        round(avg("total_experience"), 1).alias("avg_total_experience"),
        round(avg("total_compensation"), 2).alias("avg_compensation"),
        sum(when(col("IS_ADMIN") == True, 1).otherwise(0)).alias("admin_count"),
        sum(when(col("IS_TEACHER") == True, 1).otherwise(0)).alias("teacher_count"),
        countDistinct("HIGHEST_EARNED_DEGREE").alias("degree_diversity")   # Variety of educational backgrounds
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("DISTRICT_NAME",
        # Custom sort: Entry -> Junior -> Mid-Level -> Mid-Senior -> Senior
        when(col("seniority_level").contains("Entry"), 1)
        .when(col("seniority_level").contains("Junior"), 2)
        .when(col("seniority_level").contains("Mid-Level"), 3)
        .when(col("seniority_level").contains("Mid-Senior"), 4)
        .otherwise(5))

# Step 2 — Persist to gold Delta table
df_promotion_pipeline.write.format("delta").mode("overwrite").save(f"{gold_path}/promotion_pipeline")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.promotion_pipeline
    USING DELTA
    LOCATION '{gold_path}/promotion_pipeline'
""")

# Step 3 — Display the promotion funnel
display(df_promotion_pipeline)

   
---
## Scenario 3 — Admin vs Teacher Pay Equity
**Business Question:** Is there a significant compensation gap between administrators and teachers within each district? Who earns more for similar experience?

**Why it matters:** Pay equity is a key factor in teacher retention and morale. Large gaps between admin and teacher compensation can signal structural imbalances that need policy attention. Districts flagged as "Critical" or "Significant" may want to review their salary schedules.

**Approach:**
- Compute average compensation and tenure separately for admins and teachers per district.
- Join the two aggregates and calculate the absolute and percentage pay gap.
- Flag districts by severity: *Equitable* → *Moderate* → *Significant* → *Critical*.

**Output table:** `gold_employee.admin_vs_teacher_pay_equity`

In [0]:
# ============================================================
# SCENARIO 3: Admin vs Teacher Pay Equity
# ============================================================
# Computes avg compensation for admins and teachers separately,
# then joins them to quantify the pay gap per district.

# Step 1 — Admin-side aggregates
df_admin_pay = gold_emp.filter(col("IS_ADMIN") == True) \
    .groupBy("DISTRICT_NAME") \
    .agg(
        count("EMP_ID").alias("admin_headcount"),
        round(avg("total_compensation"), 2).alias("avg_admin_compensation"),
        round(avg("years_of_service"), 1).alias("avg_admin_tenure")
    )

# Step 2 — Teacher-side aggregates
df_teacher_pay = gold_emp.filter(col("IS_TEACHER") == True) \
    .groupBy("DISTRICT_NAME") \
    .agg(
        count("EMP_ID").alias("teacher_headcount"),
        round(avg("total_compensation"), 2).alias("avg_teacher_compensation"),
        round(avg("years_of_service"), 1).alias("avg_teacher_tenure")
    )

# Step 3 — Join and compute the gap
# We use an outer join to keep districts that may only have one role type.
df_pay_equity = df_admin_pay.join(df_teacher_pay, "DISTRICT_NAME", "outer") \
    .withColumn("pay_gap",                                         # Absolute dollar gap
        round(col("avg_admin_compensation") - col("avg_teacher_compensation"), 2)) \
    .withColumn("pay_gap_pct",                                     # Percentage gap relative to teacher pay
        round((col("avg_admin_compensation") - col("avg_teacher_compensation"))
              / col("avg_teacher_compensation") * 100, 1)) \
    .withColumn("equity_flag",                                     # Severity classification
        when(abs(col("pay_gap_pct")) > 50, lit("Critical Gap"))
        .when(abs(col("pay_gap_pct")) > 25, lit("Significant Gap"))
        .when(abs(col("pay_gap_pct")) > 10, lit("Moderate Gap"))
        .otherwise(lit("Equitable"))) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy(desc("pay_gap_pct"))                                  # Worst gaps first

# Step 4 — Persist to gold Delta table
df_pay_equity.write.format("delta").mode("overwrite").save(f"{gold_path}/admin_vs_teacher_pay_equity")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.admin_vs_teacher_pay_equity
    USING DELTA
    LOCATION '{gold_path}/admin_vs_teacher_pay_equity'
""")

# Step 5 — Display results sorted by largest gap
display(df_pay_equity)

   
---
## Scenario 4 — Overtime Cost Hotspots
**Business Question:** Which employees and districts have disproportionately high overtime costs? Where should the budget office investigate?

**Why it matters:** Excessive overtime is often a symptom of understaffing, poor scheduling, or role misalignment. Identifying the employees and districts with the highest overtime-to-regular-pay ratios helps administrators target audits and reduce unnecessary labor spend.

**Approach:**
- Filter employees with non-zero overtime pay.
- Calculate overtime as a percentage of regular pay.
- Classify risk levels: *Low* (<10%), *Medium* (10–20%), *High* (>20%).

**Output table:** `gold_employee.overtime_cost_hotspots`

In [0]:
# ============================================================
# SCENARIO 4: Overtime Cost Hotspots
# ============================================================
# Flags employees whose overtime pay is disproportionately high
# relative to their regular pay — a signal for potential staffing issues.

# Step 1 — Filter to employees with actual overtime and compute the ratio
df_ot_hotspots = gold_emp \
    .filter(col("OVERTIME_PAY").isNotNull() & (col("OVERTIME_PAY") > 0)) \
    .withColumn("ot_pct_of_reg",                                # OT as a % of regular pay
        round(col("OVERTIME_PAY") / col("REG_PAY") * 100, 2)) \
    .withColumn("ot_risk_level",                                # Risk classification
        when(col("ot_pct_of_reg") > 20, lit("High"))            # >20%  → investigate immediately
        .when(col("ot_pct_of_reg") > 10, lit("Medium"))         # 10-20% → monitor
        .otherwise(lit("Low"))) \
    .select(
        "EMP_ID", "EMPLOYEE_NAME", "DISTRICT_NAME",
        "REG_PAY", "OVERTIME_PAY", "ot_pct_of_reg", "ot_risk_level",
        "years_of_service", "seniority_level", "IS_ADMIN", "IS_TEACHER"
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy(desc("ot_pct_of_reg"))                             # Highest OT ratio first

# Step 2 — Persist to gold Delta table
df_ot_hotspots.write.format("delta").mode("overwrite").save(f"{gold_path}/overtime_cost_hotspots")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.overtime_cost_hotspots
    USING DELTA
    LOCATION '{gold_path}/overtime_cost_hotspots'
""")

# Step 3 — Summary stats and preview
print(f"Employees with overtime: {df_ot_hotspots.count():,}")
print(f"High-risk OT employees: {df_ot_hotspots.filter(col('ot_risk_level') == 'High').count():,}")
display(df_ot_hotspots)

   
---
## Scenario 5 — Certification Gap Report (Compliance Risk)
**Business Question:** Which teachers have expired certifications and need immediate renewal? What is the compliance risk per district?

**Why it matters:** State regulations require teachers to maintain valid certifications. Expired certs expose districts to audit findings, potential fines, and reputational risk. This report helps compliance officers prioritize renewals by urgency.

**Approach:**
- Filter the certification table to `cert_status = "Expired"`.
- Compute `days_since_expiry` and assign urgency bands (Critical 5+ yrs, High 2–5 yrs, Medium 1–2 yrs, Low <1 yr).
- Roll up to district level for a compliance summary alongside the detailed list.

**Output table:** `gold_employee.certification_gap_report`

In [0]:
# ============================================================
# SCENARIO 5: Certification Gap Report (Compliance Risk)
# ============================================================
# Identifies teachers with expired certifications and computes
# the urgency of renewal. Also produces a district-level summary.

# Step 1 — Detailed expired-cert list with urgency classification
df_cert_gap = gold_cert \
    .filter(col("cert_status") == "Expired") \
    .withColumn("days_since_expiry",                                    # How long has it been expired?
        datediff(current_date(), col("DATE_EXPIRES"))) \
    .withColumn("urgency",                                              # Urgency bands
        when(col("days_since_expiry") > 1825, lit("Critical (5+ yrs expired)"))   # ≈5 years
        .when(col("days_since_expiry") > 730,  lit("High (2-5 yrs expired)"))
        .when(col("days_since_expiry") > 365,  lit("Medium (1-2 yrs expired)"))
        .otherwise(lit("Low (<1 yr expired)"))) \
    .select(
        "EMP_ID", "EMPLOYEE_NAME", "DISTRICT_NAME",
        "CERT_DESC", "STATE_CERT_CODE",
        "DATE_EFFECTIVE", "DATE_EXPIRES",
        "days_since_expiry", "urgency"
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy(desc("days_since_expiry"))                                 # Most overdue first

# Step 2 — Roll up to district level for the compliance summary
df_cert_compliance = df_cert_gap \
    .groupBy("DISTRICT_NAME") \
    .agg(
        count("*").alias("total_expired_certs"),                         # Total expired cert records
        countDistinct("EMP_ID").alias("affected_teachers"),              # Unique teachers affected
        sum(when(col("urgency").contains("Critical"), 1).otherwise(0)).alias("critical_count"),
        sum(when(col("urgency").contains("High"), 1).otherwise(0)).alias("high_count"),
        round(avg("days_since_expiry"), 0).alias("avg_days_expired")
    ) \
    .orderBy(desc("critical_count"))                                    # Worst districts first

# Step 3 — Persist the detailed report to gold
df_cert_gap.write.format("delta").mode("overwrite").save(f"{gold_path}/certification_gap_report")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.certification_gap_report
    USING DELTA
    LOCATION '{gold_path}/certification_gap_report'
""")

# Step 4 — Display both views: district summary + detailed list
print("=== District Compliance Summary ===")
display(df_cert_compliance)
print("\n=== Detailed Expired Certifications ===")
display(df_cert_gap)

   
---
## Scenario 6 — Underpaid Senior Employees (Retention Risk)
**Business Question:** Which long-tenured employees earn below the district median? These are flight risks who may leave for better-paying districts.

**Why it matters:** Losing a 15+ year veteran is expensive — not just in recruitment costs but in institutional knowledge. If senior employees discover they're paid below median, they have both the experience and the network to move. This analysis flags them before that happens.

**Approach:**
- Calculate the approximate median compensation per district using `percentile_approx`.
- Identify employees with 15+ years of service whose total compensation falls below their district median.
- Classify retention risk by the size of the gap: *Critical* (>$30K below), *High*, *Medium*, *Low*.

**Output table:** `gold_employee.underpaid_senior_employees`

In [0]:
# ============================================================
# SCENARIO 6: Underpaid Senior Employees (Retention Risk)
# ============================================================
# Finds employees with 15+ years of service who earn below their
# district's median compensation — the classic retention red flag.

# Step 1 — Compute district-level median and average compensation
# Using percentile_approx for median (exact median is expensive on large data).
window_district_median = Window.partitionBy("DISTRICT_NAME")

df_with_median = gold_emp \
    .filter(col("total_compensation").isNotNull()) \
    .withColumn("district_median_comp",
        expr("percentile_approx(total_compensation, 0.5)").over(window_district_median)) \
    .withColumn("district_avg_comp",
        round(avg("total_compensation").over(window_district_median), 2))

# Step 2 — Filter to senior employees below median and classify risk
df_underpaid_seniors = df_with_median \
    .filter(
        (col("years_of_service") >= 15) &                       # At least 15 years of service
        (col("total_compensation") < col("district_median_comp"))  # Paid below their district median
    ) \
    .withColumn("comp_gap_vs_median",                           # How far below median? (negative = underpaid)
        round(col("total_compensation") - col("district_median_comp"), 2)) \
    .withColumn("retention_risk",                               # Risk severity based on gap size
        when(col("comp_gap_vs_median") < -30000, lit("Critical"))   # >$30K below median
        .when(col("comp_gap_vs_median") < -15000, lit("High"))     # $15K–$30K below
        .when(col("comp_gap_vs_median") < -5000,  lit("Medium"))   # $5K–$15K below
        .otherwise(lit("Low"))) \
    .select(
        "EMP_ID", "EMPLOYEE_NAME", "DISTRICT_NAME",
        "years_of_service", "total_experience", "seniority_level",
        "HIGHEST_EARNED_DEGREE", "IS_ADMIN", "IS_TEACHER",
        "total_compensation", "district_median_comp", "comp_gap_vs_median",
        "retention_risk"
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("comp_gap_vs_median")                              # Most underpaid first

# Step 3 — Persist to gold Delta table
df_underpaid_seniors.write.format("delta").mode("overwrite").save(f"{gold_path}/underpaid_senior_employees")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.underpaid_senior_employees
    USING DELTA
    LOCATION '{gold_path}/underpaid_senior_employees'
""")

# Step 4 — Print headline metrics and display detail
print(f"Total underpaid seniors (15+ yrs, below median): {df_underpaid_seniors.count():,}")
print(f"Critical retention risk: {df_underpaid_seniors.filter(col('retention_risk') == 'Critical').count():,}")
print(f"High retention risk: {df_underpaid_seniors.filter(col('retention_risk') == 'High').count():,}")
display(df_underpaid_seniors)

   
---
## Scenario 7 — Degree ROI Analysis
**Business Question:** Does a higher education degree actually pay off? What is the compensation premium for Master’s and Doctorate holders vs Bachelor’s?

**Why it matters:** Districts often incentivize advanced degrees through salary lane increases. This analysis quantifies whether those incentives translate into meaningful compensation differences — useful for both HR policy reviews and employees considering further education.

**Approach:**
- Group employees by `HIGHEST_EARNED_DEGREE` and compute avg compensation, tenure, and role mix.
- Establish a Bachelor’s degree baseline, then calculate the absolute and percentage premium for each degree level.

**Output table:** `gold_employee.degree_roi_analysis`

In [0]:
# ============================================================
# SCENARIO 7: Degree ROI Analysis
# ============================================================
# Answers: "Is a Master's or Doctorate actually worth it in terms
# of compensation?" by comparing each degree level to a Bachelor's baseline.

# Step 1 — Aggregate compensation stats by degree level
df_degree_roi = gold_emp \
    .filter(col("total_compensation").isNotNull()) \
    .groupBy("HIGHEST_EARNED_DEGREE") \
    .agg(
        count("EMP_ID").alias("employee_count"),
        round(avg("total_compensation"), 2).alias("avg_total_compensation"),
        round(avg("REG_PAY"), 2).alias("avg_reg_pay"),
        round(avg("TOTAL_BENEFITS"), 2).alias("avg_benefits"),
        round(avg("years_of_service"), 1).alias("avg_years_of_service"),
        round(avg("total_experience"), 1).alias("avg_total_experience"),
        sum(when(col("IS_ADMIN") == True, 1).otherwise(0)).alias("admin_count"),
        sum(when(col("IS_TEACHER") == True, 1).otherwise(0)).alias("teacher_count")
    ) \
    .withColumn("pct_admin",                                    # What % of this degree group are admins?
        round(col("admin_count") / col("employee_count") * 100, 1))

# Step 2 — Establish the Bachelor's baseline for comparison
# We pull the average compensation for Bachelor's holders as a scalar.
bachelor_avg = gold_emp.filter(
    (col("HIGHEST_EARNED_DEGREE").contains("Bachelor")) & col("total_compensation").isNotNull()
).agg(round(avg("total_compensation"), 2)).collect()[0][0]

# Step 3 — Calculate absolute and percentage premium vs Bachelor's
df_degree_roi = df_degree_roi \
    .withColumn("bachelor_baseline", lit(bachelor_avg)) \
    .withColumn("premium_vs_bachelor",                          # $ premium over Bachelor's avg
        round(col("avg_total_compensation") - col("bachelor_baseline"), 2)) \
    .withColumn("premium_pct",                                  # % premium over Bachelor's avg
        round((col("avg_total_compensation") - col("bachelor_baseline")) / col("bachelor_baseline") * 100, 1)) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy(desc("avg_total_compensation"))                    # Highest-paying degree first

# Step 4 — Persist to gold Delta table
df_degree_roi.write.format("delta").mode("overwrite").save(f"{gold_path}/degree_roi_analysis")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.degree_roi_analysis
    USING DELTA
    LOCATION '{gold_path}/degree_roi_analysis'
""")

# Step 5 — Display with baseline context
print(f"Bachelor baseline avg compensation: ${bachelor_avg:,.2f}")
display(df_degree_roi)

   
---
## Scenario 8 — Quarterly Pay Breakdown (Line Item Detail)
**Business Question:** How is employee compensation distributed across quarters? Which pay types (regular, overtime, bonus, benefits) dominate each quarter?

**Why it matters:** Compensation doesn’t flow evenly through the year — bonuses spike in certain quarters, benefits renew annually, and overtime may cluster around school events. Understanding the quarterly rhythm helps finance teams with cash-flow planning and budget forecasting.

**Approach:**
- Join the silver-layer `fact_pab_lineitem` with `dim_pab_item` (for pay-type descriptions) and `fact_pab` (for employee linkage).
- Extract the quarter from `BEG_DATE` and aggregate amounts by pay type and quarter.

**Output table:** `gold_employee.quarterly_pay_breakdown`

In [0]:
# ============================================================
# SCENARIO 8: Quarterly Pay Breakdown (Line Item Detail)
# ============================================================
# Joins silver-layer fact/dim tables to show how compensation
# flows across quarters and pay types (regular, OT, benefits, etc.).

# Step 1 — Join line items → item descriptions → employee linkage
df_quarterly = fact_lineitem.alias("li") \
    .join(dim_pab_item.alias("pi"),                            # Get pay-type descriptions
          col("li.PAB_ITEM_ID") == col("pi.PAB_ITEM_ID"), "left") \
    .join(
        fact_pab.select("PAB_ID", "EMP_ID").alias("p"),        # Link back to employee
        col("li.PAB_ID") == col("p.PAB_ID"),
        "left"
    ) \
    .withColumn("quarter",                                     # Extract calendar quarter from begin date
        concat(lit("Q"), quarter(col("li.BEG_DATE"))))

# Step 2 — Aggregate by pay type and quarter
df_qtr_breakdown = df_quarterly \
    .groupBy(
        trim(col("pi.TYPE")).alias("pay_type"),                 # Clean up whitespace in type labels
        "quarter"
    ) \
    .agg(
        count("*").alias("line_item_count"),                    # Volume of line items
        round(sum("AMOUNT_POSTED"), 2).alias("total_amount"),   # Total dollars
        round(avg("AMOUNT_POSTED"), 2).alias("avg_amount"),     # Average per line item
        round(min("AMOUNT_POSTED"), 2).alias("min_amount"),
        round(max("AMOUNT_POSTED"), 2).alias("max_amount"),
        countDistinct(col("p.EMP_ID")).alias("employee_count")  # Unique employees in this bucket
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("pay_type", "quarter")

# Step 3 — Persist to gold Delta table
df_qtr_breakdown.write.format("delta").mode("overwrite").save(f"{gold_path}/quarterly_pay_breakdown")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.quarterly_pay_breakdown
    USING DELTA
    LOCATION '{gold_path}/quarterly_pay_breakdown'
""")

# Step 4 — Display the quarterly breakdown
display(df_qtr_breakdown)

   
---
## Scenario 9 — Benefits Cost Optimization
**Business Question:** Which employees have disproportionately high or low benefits relative to their compensation? Where can the district optimize benefit spending?

**Why it matters:** Benefits typically represent 20–30% of total compensation. Employees with benefits far above or below this range may indicate data issues, policy exceptions, or optimization opportunities. Districts can use this to benchmark benefit structures against peers.

**Approach:**
- Compute `benefits_pct` = TOTAL_BENEFITS / total_compensation for each employee.
- Bucket employees into bands: *Very High* (>30%), *High* (20–30%), *Normal* (10–20%), *Low* (<10%), *No Benefits*.
- Aggregate by district and band for a cost-optimization summary.

**Output table:** `gold_employee.benefits_cost_optimization`

In [0]:
# ============================================================
# SCENARIO 9: Benefits Cost Optimization
# ============================================================
# Computes benefits as a percentage of total compensation and flags
# outliers for districts to investigate (too high or suspiciously low).

# Step 1 — Calculate benefits % and classify into bands
df_benefits = gold_emp \
    .filter(col("total_compensation").isNotNull() & (col("total_compensation") > 0)) \
    .withColumn("benefits_pct",                                 # Benefits share of total comp
        round(col("TOTAL_BENEFITS") / col("total_compensation") * 100, 1)) \
    .withColumn("benefits_flag",                                # Band classification
        when(col("benefits_pct") > 30, lit("Very High (>30%)"))    # Potential cost concern
        .when(col("benefits_pct") > 20, lit("High (20-30%)"))
        .when(col("benefits_pct") > 10, lit("Normal (10-20%)"))    # Industry standard range
        .when(col("benefits_pct") > 0,  lit("Low (<10%)"))         # Possibly underinsured
        .otherwise(lit("No Benefits")))

# Step 2 — Summarize by district and benefit band
df_benefits_summary = df_benefits \
    .groupBy("DISTRICT_NAME", "benefits_flag") \
    .agg(
        count("EMP_ID").alias("employee_count"),
        round(avg("benefits_pct"), 1).alias("avg_benefits_pct"),
        round(sum("TOTAL_BENEFITS"), 2).alias("total_benefits_spend"),
        round(avg("total_compensation"), 2).alias("avg_compensation")
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("DISTRICT_NAME",
        # Custom sort so "Very High" appears first within each district
        when(col("benefits_flag").contains("Very High"), 1)
        .when(col("benefits_flag").contains("High ("), 2)
        .when(col("benefits_flag").contains("Normal"), 3)
        .when(col("benefits_flag").contains("Low"), 4)
        .otherwise(5))

# Step 3 — Persist to gold Delta table
df_benefits_summary.write.format("delta").mode("overwrite").save(f"{gold_path}/benefits_cost_optimization")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.benefits_cost_optimization
    USING DELTA
    LOCATION '{gold_path}/benefits_cost_optimization'
""")

# Step 4 — Display the optimization summary
display(df_benefits_summary)

   
---
## Scenario 10 — Retirement Risk & Workforce Planning
**Business Question:** Which employees are approaching retirement (30+ years of service)? How many will the district need to replace in the next 5 years, and what is the budget impact?

**Why it matters:** A wave of retirements without a succession plan can cripple a district’s operations. Knowing *how many* and *in which roles* retirees sit helps HR build a proactive hiring and mentoring pipeline.

**Approach:**
- Band employees by years of service: *Imminent* (35+), *Near-term* (30–34), *Mid-term* (25–29), *Long-term* (<25).
- Summarize per district with headcounts, role splits (teacher vs admin), and total compensation at risk.

**Output table:** `gold_employee.retirement_risk_analysis`

In [0]:
# ============================================================
# SCENARIO 10: Retirement Risk & Workforce Planning
# ============================================================
# Segments employees into retirement bands by years of service
# and quantifies the budget impact of upcoming retirements per district.

# Step 1 — Assign retirement-proximity bands
df_retirement = gold_emp \
    .filter(col("total_compensation").isNotNull()) \
    .withColumn("retirement_band",
        when(col("years_of_service") >= 35, lit("Imminent (35+ yrs)"))    # Could retire any day
        .when(col("years_of_service") >= 30, lit("Near-term (30-34 yrs)"))  # Within ~5 years
        .when(col("years_of_service") >= 25, lit("Mid-term (25-29 yrs)"))   # Plan for succession
        .otherwise(lit("Long-term (<25 yrs)")))

# Step 2 — District-level retirement risk summary
df_retirement_risk = df_retirement \
    .groupBy("DISTRICT_NAME", "retirement_band") \
    .agg(
        count("EMP_ID").alias("employee_count"),
        sum(when(col("IS_TEACHER") == True, 1).otherwise(0)).alias("teacher_count"),
        sum(when(col("IS_ADMIN") == True, 1).otherwise(0)).alias("admin_count"),
        round(sum("total_compensation"), 2).alias("total_comp_at_risk"),   # Budget impact if all retire
        round(avg("total_compensation"), 2).alias("avg_compensation"),
        round(avg("years_of_service"), 1).alias("avg_tenure")
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("DISTRICT_NAME",
        # Sort so imminent retirements appear first
        when(col("retirement_band").contains("Imminent"), 1)
        .when(col("retirement_band").contains("Near-term"), 2)
        .when(col("retirement_band").contains("Mid-term"), 3)
        .otherwise(4))

# Step 3 — Persist to gold Delta table
df_retirement_risk.write.format("delta").mode("overwrite").save(f"{gold_path}/retirement_risk_analysis")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.retirement_risk_analysis
    USING DELTA
    LOCATION '{gold_path}/retirement_risk_analysis'
""")

# Step 4 — High-level retirement headline numbers
imminent = df_retirement.filter(col("retirement_band").contains("Imminent")).count()
near_term = df_retirement.filter(col("retirement_band").contains("Near-term")).count()
print(f"Imminent retirees (35+ yrs): {imminent:,}")
print(f"Near-term retirees (30-34 yrs): {near_term:,}")
display(df_retirement_risk)

   
---
## Scenario 11 — Teacher Certification Breadth
**Business Question:** Which teachers hold the most diverse certifications? Do multi-certified teachers earn more, and are they concentrated in specific districts?

**Why it matters:** Teachers with broad certification portfolios are more versatile — they can cover multiple subjects, adapt to curriculum changes, and are generally harder to replace. Understanding whether the market rewards this breadth helps districts design better incentive structures.

**Approach:**
- Count total, active, and expired certifications per teacher from `gold_certification_overview`.
- Join back to the employee summary to correlate certification breadth with compensation and tenure.
- Rank teachers by total certs (tie-broken by compensation) for a "most versatile" leaderboard.

**Output table:** `gold_employee.teacher_certification_breadth`

In [0]:
# ============================================================
# SCENARIO 11: Teacher Certification Breadth
# ============================================================
# Counts the number and diversity of certifications per teacher,
# then correlates breadth with compensation to test whether
# multi-certified teachers earn more.

# Step 1 — Aggregate certification metrics per teacher
df_cert_count = gold_cert \
    .groupBy("EMP_ID") \
    .agg(
        count("*").alias("total_certs"),                                # Total certs (active + expired)
        sum(when(col("is_active") == True, 1).otherwise(0)).alias("active_certs"),
        sum(when(col("cert_status") == "Expired", 1).otherwise(0)).alias("expired_certs"),
        collect_set("CERT_DESC").alias("cert_list"),                    # Distinct cert descriptions
        round(avg("cert_duration_days"), 0).alias("avg_cert_duration_days")
    )

# Step 2 — Join certification counts back to employee demographics
df_cert_breadth = df_cert_count.alias("cc") \
    .join(
        gold_emp.select(
            "EMP_ID", "EMPLOYEE_NAME", "DISTRICT_NAME",
            "years_of_service", "total_compensation", "HIGHEST_EARNED_DEGREE"
        ).alias("e"),
        col("cc.EMP_ID") == col("e.EMP_ID"),
        "left"
    ) \
    .select(
        col("e.EMP_ID"),
        col("e.EMPLOYEE_NAME"),
        col("e.DISTRICT_NAME"),
        "total_certs", "active_certs", "expired_certs",
        "avg_cert_duration_days",
        col("e.HIGHEST_EARNED_DEGREE"),
        col("e.years_of_service"),
        col("e.total_compensation"),
        "cert_list"
    ) \
    .withColumn("cert_breadth_rank",                                   # Rank: most certs first, break ties by pay
        rank().over(Window.orderBy(desc("total_certs"), desc("total_compensation")))) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("cert_breadth_rank")

# Step 3 — Persist to gold Delta table
df_cert_breadth.write.format("delta").mode("overwrite").save(f"{gold_path}/teacher_certification_breadth")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.teacher_certification_breadth
    USING DELTA
    LOCATION '{gold_path}/teacher_certification_breadth'
""")

# Step 4 — Summary stats and preview
print(f"Teachers with certifications: {df_cert_breadth.count():,}")
print(f"Avg certs per teacher: {df_cert_count.agg(round(avg('total_certs'), 1)).collect()[0][0]}")
display(df_cert_breadth)